In [2]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import psycopg2
from sqlalchemy import create_engine
from datetime import datetime

In [3]:
load_dotenv()

True

In [4]:


API_KEY = os.getenv("API_KEY")

def extract_news():

    url = f"https://newsapi.org/v2/top-headlines?country=us&apiKey={API_KEY}"
    
    response = requests.get(url)

    response.raise_for_status()

    data = response.json()

    articles = data.get("articles",[])

    return articles

data = extract_news()

print(data)


[{'source': {'id': None, 'name': 'NPR'}, 'author': 'Bill Chappell', 'title': "Alex Murdaugh will get a new murder trial. Here's a timeline of his case - NPR", 'description': 'Alex Murdaugh — the disgraced former lawyer serving a life term for the murders of his wife and son — will get a new trial in South Carolina, the state Supreme Court said on Wednesday.', 'url': 'https://www.npr.org/2026/05/13/nx-s1-5719271/alex-murdaugh-murder-timeline-trial', 'urlToImage': 'https://npr.brightspotcdn.com/dims3/default/strip/false/crop/2477x1393+0+329/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2Fcd%2F3e%2F5ebac9e8499facbb63fbb9872737%2Fap25337691935332.jpg', 'publishedAt': '2026-05-13T17:14:32Z', 'content': "Three years after Alex Murdaugh's double-murder trial ended with two life sentences, the South Carolina Supreme Court on Wednesday granted him a new trial in the killings of his wife and son. Murdaug… [+11365 chars]"}, {'source': {'id': 'cbs-news', 'nam

In [5]:
def transform_news(data):
    
    cleaned_data = []
    
    for article in data:
        cleaned_data.append(
            {
                "source"       : article.get("source",{}).get("name"),
                "author"       : article.get("author"),
                "title"        : article.get("title"),
                "published_at" : article.get("publishedAt"),
                "inserted_at"  : datetime.now() 
            }
        )
    
    cleaned_df = pd.DataFrame(cleaned_data)

    return cleaned_df

transformed_df = transform_news(data)

print(transformed_df.head())
print(transformed_df)


           source                            author  \
0             NPR                     Bill Chappell   
1        CBS News  Jennifer  Jacobs, James  LaPorta   
2  9to5google.com                          Abner Li   
3        CBS News                     Kaia  Hubbard   
4         9to5Mac                          Zac Hall   

                                               title          published_at  \
0  Alex Murdaugh will get a new murder trial. Her...  2026-05-13T17:14:32Z   
1  Netanyahu made secret visit to UAE to meet wit...  2026-05-13T17:10:00Z   
2  Google on its ‘continued commitment to Chromeb...  2026-05-13T16:45:00Z   
3  Senate defeats 7th attempt to limit Trump's Ir...  2026-05-13T16:32:00Z   
4  Meta launches Instants, a new iPhone app and I...  2026-05-13T16:19:00Z   

                 inserted_at  
0 2026-05-14 23:03:42.085131  
1 2026-05-14 23:03:42.085138  
2 2026-05-14 23:03:42.085140  
3 2026-05-14 23:03:42.085141  
4 2026-05-14 23:03:42.085143  
            so

In [6]:
def load_to_postgres(transformed_df):

    user     = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    host     =  os.getenv("DB_HOST")
    port     =  os.getenv("DB_PORT")
    dbname   = os.getenv("DB_NAME")

    engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require')

    transformed_df.to_sql(name="news",con=engine, if_exists="append",index=False)

    print(f"LOAD SUCCESSFUL: {len(transformed_df)} rows added to DB")


load_to_postgres(transformed_df)


LOAD SUCCESSFUL: 17 rows added to DB


In [7]:
from dotenv import load_dotenv
from extract import extract_news
from transform import transform_news
from load import load_to_postgres

load_dotenv()

def news_etl():
    print("--- Starting Tesla News ETL Pipeline ---")

    try:
        # 1. EXTRACT
        print("Step 1: Extracting data from NewsAPI...")
        raw_data = extract_news()
        print(f"Successfully retrieved {len(raw_data)} articles.")

        # 2. TRANSFORM
        print("Step 2: Transforming data into DataFrame...")
        
        transformed_df = transform_news(raw_data) 
        
        if transformed_df.empty:
            print("No data found to transform. Exiting.")
            return

        # 3. LOAD
        print("Step 3: Loading data into PostgreSQL...")
        load_to_postgres(transformed_df)
        
        print("--- ETL Pipeline Completed Successfully ---")

    except Exception as e:
        print(f"!!! ETL Pipeline Failed: {e}")

if __name__ == "__main__":
    news_etl()


[{'source': {'id': None, 'name': 'NPR'}, 'author': 'Bill Chappell', 'title': "Alex Murdaugh will get a new murder trial. Here's a timeline of his case - NPR", 'description': 'Alex Murdaugh — the disgraced former lawyer serving a life term for the murders of his wife and son — will get a new trial in South Carolina, the state Supreme Court said on Wednesday.', 'url': 'https://www.npr.org/2026/05/13/nx-s1-5719271/alex-murdaugh-murder-timeline-trial', 'urlToImage': 'https://npr.brightspotcdn.com/dims3/default/strip/false/crop/2477x1393+0+329/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2Fcd%2F3e%2F5ebac9e8499facbb63fbb9872737%2Fap25337691935332.jpg', 'publishedAt': '2026-05-13T17:14:32Z', 'content': "Three years after Alex Murdaugh's double-murder trial ended with two life sentences, the South Carolina Supreme Court on Wednesday granted him a new trial in the killings of his wife and son. Murdaug… [+11365 chars]"}, {'source': {'id': 'cbs-news', 'nam

In [10]:
import os
import requests
import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine


load_dotenv()

API_KEY = os.getenv("API_KEY")



def news_etl_pipeline():
    """
    Full ETL pipeline with nested functions
    """
    # EXTRACT

    def extract():
        print("Extracting data...")

        url = f"https://newsapi.org/v2/top-headlines?country=us&apiKey={API_KEY}"

        response = requests.get(url)

        response.raise_for_status()

        data = response.json()
        
        articles = data.get("articles", [])
        
        return articles

    # TRANSFORM

    def transform(articles):
        print("Transforming data...")

        cleaned_data = []

        for article in articles:
            cleaned_data.append(
                {
                    "source": article.get("source", {}).get("name"),
                    "author": article.get("author"),
                    "title": article.get("title"),
                    "published_at": article.get("publishedAt"),
                    "inserted_at": datetime.now()
                }
            )


        cleaned_df = pd.DataFrame(cleaned_data)

        return cleaned_df

    # LOAD

    def load(df):
        print("Loading data into PostgreSQL...")

        user = os.getenv("DB_USER")
        password = os.getenv("DB_PASSWORD")
        host = os.getenv("DB_HOST")
        port = os.getenv("DB_PORT")
        dbname = os.getenv("DB_NAME")

        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")

        df.to_sql("news",con=engine,if_exists="append",index=False)

        print(f"LOAD SUCCESSFUL: {len(df)} rows inserted")

    # PIPELINE EXECUTION

    print("Starting ETL Pipeline...")

    articles = extract()
    df = transform(articles)
    load(df)

    print("ETL Pipeline Completed Successfully!")


# Run pipeline
if __name__ == "__main__":
    news_etl_pipeline()


Starting ETL Pipeline...
Extracting data...
Transforming data...
Loading data into PostgreSQL...
LOAD SUCCESSFUL: 17 rows inserted
ETL Pipeline Completed Successfully!
